# 📢 Vietnamese Advertisement Generator — Inference

Generate Vietnamese product ads using the fine-tuned **Qwen3-0.6B + LoRA** model.

**Model**: [`vmhdaica/advertisement-lora`](https://huggingface.co/vmhdaica/advertisement-lora)  
**Base**: [`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B)  
**Runtime**: GPU T4 (free Colab) — ~60-80s per ad

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate torch sentencepiece

## 2. Load Model + LoRA Adapter

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-0.6B"
ADAPTER = "vmhdaica/advertisement-lora"

print(f"Loading base model: {BASE_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading LoRA adapter: {ADAPTER}")
model = PeftModel.from_pretrained(base_model, ADAPTER)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.eval()
print(f"\n✅ Model loaded on {model.device}")
print(f"   Vocab size: {len(tokenizer):,}")

## 3. Generate Advertisement

Edit the `product_name` and `description` below with your own product info.

In [ ]:
def generate_ad(
    product_name: str,
    description: str,
    max_new_tokens: int = 1024,
    num_beams: int = 2,
    temperature: float = 0.7,
    top_p: float = 0.9,
    no_repeat_ngram_size: int = 3,
) -> str:
    """Generate a Vietnamese advertisement for the given product."""
    user_content = (
        f"tạo quảng cáo cho sản phẩm sau:\n"
        f"Tên sản phẩm: {product_name}\n"
        f"Mô tả: {description}"
    )
    prompt = f"<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=2048 - max_new_tokens,
        add_special_tokens=False,
    ).to(model.device)

    # Stop at <|im_end|> or EOS
    eos_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id != tokenizer.unk_token_id:
        eos_ids.append(im_end_id)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_ids,
        )

    input_len = inputs["input_ids"].shape[1]
    generated = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    return generated.strip()

### Example 1

In [ ]:
# ✏️ Edit these with your product info
product_name = "Áo thun nam thể thao chạy bộ thoáng khí PROMAX"
description = "Chất liệu vải mè siêu nhẹ, co giãn 4 chiều, thấm hút mồ hôi. Thiết kế form regular fit, phù hợp chạy bộ, tập gym."

ad = generate_ad(product_name, description)
print(ad)

### Example 2

In [ ]:
product_name = "Nồi chiên không dầu Xiaomi Smart Air Fryer 6.5L"
description = "Dung tích 6.5L, công suất 1800W, điều khiển qua app Mi Home, 8 chế độ nấu tự động, lớp phủ chống dính."

ad = generate_ad(product_name, description)
print(ad)

## 4. Batch Generation from CSV

Upload a CSV file with `product_name` and `description` columns to generate ads for multiple products at once.

In [ ]:
import pandas as pd
import time

def generate_ads_from_csv(csv_path: str, output_path: str = "output_ads.csv"):
    """Generate ads for all products in a CSV file."""
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} products from {csv_path}")

    ads = []
    for i, row in df.iterrows():
        name = str(row.get("product_name", ""))
        desc = str(row.get("cleaned_description", row.get("description", "")))
        print(f"\n[{i+1}/{len(df)}] {name[:60]}...")

        t0 = time.time()
        ad = generate_ad(name, desc)
        elapsed = time.time() - t0

        ads.append(ad)
        print(f"  ✅ Done in {elapsed:.1f}s ({len(ad)} chars)")

    df["generated_advertisement"] = ads
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"\n📁 Saved to {output_path}")
    return df

# Usage: upload your CSV and run:
# result_df = generate_ads_from_csv("your_products.csv")

## 5. Generation Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `max_new_tokens` | 1024 | Maximum tokens to generate |
| `num_beams` | 2 | Beam search width (higher = better quality, slower) |
| `temperature` | 0.7 | Randomness (lower = more deterministic) |
| `top_p` | 0.9 | Nucleus sampling threshold |
| `no_repeat_ngram_size` | 3 | Prevent repeating n-grams |